# RAG + Langgraph
 - The LLM's role here is purely reading and summarizing. It never decided to search anything. It doesn't know ChromaDB exists. 
 - It doesn't know it received retrieved chunks vs hardcoded text. You could have pasted the context manually and it would behave identically.

In [1]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader,PyMuPDFLoader,DirectoryLoader
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from langchain_groq import ChatGroq
import uuid
from typing import TypedDict, List,Dict,Any
import numpy as np
 
import os

/Users/nisargbhatia/Desktop/langGraph/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import sys
print(sys.executable)

In [2]:
#pdf loader

dirloader_pdf=DirectoryLoader('../data/pdf/',glob='**/*.pdf',loader_cls=PyMuPDFLoader)
docs_pdf=dirloader_pdf.load()
print(docs_pdf)

[Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2025-07-25T11:02:56+05:30', 'source': '../data/pdf/IT314-Software Engineering-SPM Cont.pdf', 'file_path': '../data/pdf/IT314-Software Engineering-SPM Cont.pdf', 'total_pages': 10, 'format': 'PDF 1.5', 'title': 'Microsoft PowerPoint - IT314-Software Engineering-SPM Cont.ppt [Compatibility Mode]', 'author': 'DA-IICT', 'subject': '', 'keywords': '', 'moddate': '2025-07-25T11:02:56+05:30', 'trapped': '', 'modDate': "D:20250725110256+05'30'", 'creationDate': "D:20250725110256+05'30'", 'page': 0}, page_content='7/25/2025\n1\nDA-IICT\nIT 314: Software Engineering\nSoftware Process Models – RUP|XP|TDD\n1\nRUP – Rational Unified Process\n•\nLife Cycle model proposed by Booch, Jacobson, and Rumbaugh\n(“The three Amigos”) derived from the work on UML\n•\nRational Unified Process (RUP) uses Unified Modeling Language\n(UML) as core notation\n•\nDescribed from 3 perspective

In [3]:
class embedding:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            self.model=SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully, with dimension {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")

    def generate_embedding(self,texts:List[str])->np.ndarray:
        if self.model is None:
            raise ValueError("Model is not loaded.")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
#initalize
embed_manager=embedding()

Model all-MiniLM-L6-v2 loaded successfully, with dimension 384


In [4]:
### Text splitting get into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance
         '''Why recursive? It tries separators from most-natural to least. 
        It first tries to split at \n\n (paragraph breaks). 
        If a chunk is still too large, it splits at \n (line breaks), 
        then spaces (word boundaries), and finally at the character level as a last resort. 
        This preserves semantic coherence.'''
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""] 
       
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(docs_pdf)

Split 10 documents into 11 chunks

Example chunk:
Content: 7/25/2025
1
DA-IICT
IT 314: Software Engineering
Software Process Models – RUP|XP|TDD
1
RUP – Rational Unified Process
•
Life Cycle model proposed by Booch, Jacobson, and Rumbaugh
(“The three Amigos”)...
Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2025-07-25T11:02:56+05:30', 'source': '../data/pdf/IT314-Software Engineering-SPM Cont.pdf', 'file_path': '../data/pdf/IT314-Software Engineering-SPM Cont.pdf', 'total_pages': 10, 'format': 'PDF 1.5', 'title': 'Microsoft PowerPoint - IT314-Software Engineering-SPM Cont.ppt [Compatibility Mode]', 'author': 'DA-IICT', 'subject': '', 'keywords': '', 'moddate': '2025-07-25T11:02:56+05:30', 'trapped': '', 'modDate': "D:20250725110256+05'30'", 'creationDate': "D:20250725110256+05'30'", 'page': 0}


## Vector Store

In [6]:
class vectorstore():
    #collection means where exactly in vector store,we store our vectors
    def __init__(self,collection_name:str="default",persist_directory:str="../data/vectorstore"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_client()

    def _initialize_client(self):
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
              
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            self.collection=self.client.get_or_create_collection(name=self.collection_name,metadata={"description":"Vector store collection"})
            
            print(f"ChromaDB client initialized with collection: {self.collection_name}")
       

        except Exception as e:
            print(f"Error initializing ChromaDB client: {e}")

    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        if(len(documents)!=len(embeddings)):
            raise ValueError("Number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to vector store...")
    
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #this enumerate(zip...) fxn is used to create array of tuples(doc,embedding)
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
            
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            '''In ChromaDB, collections.add() has these arguments, 
            id,embedding list,metadatas,documents.'''
            print(f"Successfully added {len(documents)} documents to vector store")

            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vec_store=vectorstore()
vec_store

ChromaDB client initialized with collection: default


In [7]:
# Converting text to embeddings
texts=[doc.page_content for doc in chunks]

In [8]:
#To generate embeddings
embeddings=embed_manager.generate_embedding(texts) #we created embed_manager object earlier

#to store in vector store
vec_store.add_documents(chunks,embeddings)

Batches: 100%|██████████| 1/1 [00:03<00:00,  3.40s/it]

Generated embeddings with shape: (11, 384)
Adding 11 documents to vector store...
Successfully added 11 documents to vector store


# Retriever Pipeline from vector store
- Retriever is an interface built on top of the vector store, which allows us to retrieve relevant documents based on a query. 
- It uses the vector store to find the most similar documents to the query and returns them as results.

In [9]:
class retrierver:
    def __init__(self,vectorstore:vectorstore,embedding_manager:embedding):
        self.vectorstore=vectorstore
        self.embedding_manager=embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embedding([query])    
        
        # Search in vector store
        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding[0].tolist()],
                n_results=top_k
            )
            ''' THe structure of results is: 
            results= {
    'ids': [            #One entry per query
        ['doc_a1b2c3d4_0', 'doc_e5f6g7h8_1', 'doc_i9j0k1l2_2']  --> This corresponds to result of query[0]
    ],
    'documents': [
        ['Chunk text one...', 'Chunk text two...', 'Chunk text three...']
    ],
    'metadatas': [
        [
            {'source': 'file.pdf', 'page': 0, 'doc_index': 0, 'content_length': 843},
            {'source': 'file.pdf', 'page': 2, 'doc_index': 1, 'content_length': 912},
            {'source': 'other.pdf', 'page': 1, 'doc_index': 2, 'content_length': 756},
        ]
    ],
    'distances': [
        [0.12, 0.34, 0.51]    # cosine distances, ascending (smaller = more similar)
    ],
    'embeddings': None         # only populated if you pass include=['embeddings']
}
 '''




            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]    #We  sent 1 query, so the outer list always has exactly
                                                       # 1 element — at index 0
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                print(retrieved_docs)
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=retrierver(vec_store,embed_manager)                       
        

In [10]:
# rag_retriever.retrieve("XP is a lightweight methodology for small to medium ")

In [11]:
llm=ChatGroq(model='llama-3.1-8b-instant',groq_api_key=os.getenv('GROQ_API_KEY'))

In [12]:
def rag_simple(query:str,retriver,llm,top_k=5):
    results=retriver.retrieve(query,top_k=top_k)
    """retrieve returns a list of dictionaries with keys:
        'id', 'content', 'metadata', 'similarity_score', 'distance', 'rank'
       """
 

    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    prompt=f"""Use the following context to answer the question:

    Context:{context}

    Question: {query}

    Mention whether the answer was based on your pretrained knowlege or the retrieved context. 
    If based on retrieved context, mention the source document and page number from metadata.
    If empty or missing, don't expect for user to give it,
      answer based on your pretrained knowledge but mention that context 
    was insufficient.
    ANswer:
    """
    response=llm.invoke([HumanMessage(content=prompt)])
    print(f"LLM Response: {response}")
    return response

In [13]:
# res=rag_simple("Tell me something related to DAIICT college",rag_retriever,llm)
# res.pretty_print()

# Enhanced RAG Pipeline

In [14]:
def advanced_rag(query:str,retriver,llm,top_k=5,min_score=0.2,return_context=False):
    '''RAG with more features:
    returns answers,sources, similarity scores,and optionally full context
    '''
    results=retriver.retrieve(query,top_k=top_k,score_threshold=min_score)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    sources=[
        {
            "source": doc['metadata'].get('source', 'unknown'),
            'page': doc['metadata'].get('page', 'unknown'),
            'similarity_score': doc['similarity_score'],
        } for doc in results
    ]
    prompt=f"""
        Use the following context to answer the question:
        Context:{context}
        Question: {query}


    Mention whether the answer was based on your pretrained knowlege or the retrieved context. 
    If based on retrieved context, mention the source document and page number from metadata.
    Only If empty or missing, don't expect for user to give it and answer based on your pretrained knowledge but mention that context 
    was insufficient for this query. otherwise, use the context only to answer the question.


    """
    response=llm.invoke([HumanMessage(content=prompt)])
    print(f"LLM Response: {response}")
    output={
        "answer": response.content,
        "sources": sources 
    }
    if return_context:
        output['context'] = context
    return output

In [15]:
result=advanced_rag("Summarize Software Engineering-SPM Cont.pdf going through XP methodology,life of a unified Process, RUP",rag_retriever,llm,top_k=3,min_score=0.1,return_context=True)
print("LLM OUTPUT: ",result['answer'])
print()
print("Sources: ",result['sources'])
print("Context: ",result['context'])

Retrieving documents for query: 'Summarize Software Engineering-SPM Cont.pdf going through XP methodology,life of a unified Process, RUP'
Top K: 3, Score threshold: 0.1


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.50it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
[{'id': 'doc_29ec2c51_5', 'content': '7/25/2025\n5\nLife of a Unified Process\nRUP - Summary\n•\nThe RUP is not a suitable process for all types of development but it\ndoes represent a new generation of generic processes\n•\nMost important innovation:\n•\n Combination of many views\n•\n Deployment of software is part of the process (almost ignored\nin other process models)\n•\nBased on standards\n•\n Object-oriented Modeling\n•\n Unified Modeling Language', 'metadata': {'content_length': 402, 'total_pages': 10, 'page': 4, 'modDate': "D:20250725110256+05'30'", 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2025-07-25T11:02:56+05:30', 'doc_index': 5, 'creationDate': "D:20250725110256+05'30'", 'subject': '', 'moddate': '2025-07-25T11:02:56+05:30', 'file_path': '../data/pdf/IT314-Software Engineering-SPM Cont.pdf', 'author': 'DA-IICT', 'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'keywords': '

NameError: name 'HumanMessage' is not defined